In [ ]:
import json
import csv
import time
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

# ================= 1. 配置区 =================
CONFIGS = {
    "Qwen": {"api_key": "sk-xxx", "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1", "model": "qwen3.5-plus"},
}

MAX_WORKERS = 10 
JSON_FILE = "Product Description.json"
OUTPUT_CSV = "Qwen Results.csv"
ITERATIONS = 100 
MAX_RETRY_ROUNDS = 3 

# ================= 2. 提问模板 =================
GUIDE_QUERIES = {
    "Apparel": {
        "1.1": {"CN": "我想买一双日常通勤穿的运动鞋，预算中等价位，脚比较宽，想要轻便透气、走路不累的款式，有什么推荐？", "EN": "I want to buy daily sneakers, medium price, wide feet, breathable and comfortable. Any recommendations?"},
        "1.2": {"CN": "我想在 Adidas 的跑鞋里选一双，预算在 700-800 元左右，主要用于全马和半马的日常备赛训练。我比较重视推进感和抓地力，能不能帮我筛选一下并说明理由？", "EN": "I want to choose Adidas running shoes, budget 700-800, for marathon training. I value propulsion and grip. Can you recommend one?"},
        "1.3": {"CN": "我目前想在 Champion Champracer PRO1 和 Nike Air Zoom Upturn SC 之间选一双。我主要用于日常走路通勤，预算比较有限。我非常看重复古颜值和缓震感。针对性价比以及夏天不闷脚，哪个更好？", "EN": "Hesitating between Champion PRO1 and Nike Air Zoom. For commuting, limited budget. Which one is better for cost-performance and breathability?"}
    },
    "Cosmetics": {
        "1.1": {"CN": "我想买一款适合油性/混油皮肤的粉底液，要求持久控油、不容易暗沉，价格中等偏上，有什么品牌可以考虑？", "EN": "Looking for foundation for oily skin, long-lasting oil control, anti-dulling. Any suggestions?"},
        "1.2": {"CN": "我想买一款 兰蔻 的粉底液，预算 500 元以内，我属于典型的混油皮，T区经常出油且容易脱妆。我比较重视持久度和不卡粉，能帮我说明它的持妆技术吗？", "EN": "Want Lancôme foundation under 500 for oily skin. I value longevity and no-cakey. Can you explain its technology?"},
        "1.3": {"CN": "我在纠结买 香奈儿“金砖”粉底液 还是 欧莱雅“小金牌”粉底液。我属于熟龄干性皮肤，看重抗老修护。针对卸妆后是否暗沉以及减少细纹，哪一个更让你信赖？", "EN": "Chanel vs L'Oreal Golden Lift. Mature dry skin, anti-aging. Which one is more trustworthy for anti-dulling and wrinkle reduction?"}
    },
    "Electronics": {
        "1.1": {"CN": "想换一部手机，预算 4000 左右，要求拍照和电池续航要好一点，系统流畅，平时也打游戏。有什么建议？", "EN": "New phone, budget 4000, good photo and battery, smooth OS. Any advice?"},
        "1.2": {"CN": "我想买一款 小米 的高端旗舰手机，预算 5500 元左右。我平时经常自拍，且需要快速查看外卖和打车动态。听说小米有带背屏的手机，能推荐并说明理由吗？", "EN": "Want Xiaomi flagship, 5500 budget. I like selfies and need to check delivery status. Recommend one with a back screen?"},
        "1.3": {"CN": "我目前在 iPhone 17 Pro Max 和 Xiaomi 17 Pro Max 之间犹豫。我主要用于专业摄影和高强度办公，看重长焦和续航。针对出差不充电和演唱会拍摄，哪个更适合我？", "EN": "iPhone 17 PM vs Xiaomi 17 PM. Professional photo and office use. Which one is better for long trips and concert shooting?"}
    }
}

csv_lock = threading.Lock()

def fetch_ai_response(task):
    model_name, cfg, cat_key, lang_key, fmt_key, g_type, run_id, full_prompt = task
    client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])
    try:
        res = client.chat.completions.create(
            model=cfg['model'],
            messages=[{"role": "user", "content": full_prompt}],
            temperature=0.9,
            timeout=100 
        )
        content = res.choices[0].message.content
        with csv_lock:
            with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8-sig') as f:
                writer = csv.writer(f)
                writer.writerow([model_name, cat_key, lang_key, fmt_key, g_type, run_id, content])
        return True
    except Exception as e:
        # 失败时不打印冗长信息，保持界面干净
        return False

def execute_tasks_with_retry(tasks):
    """
    内部执行逻辑，保持你要求的进度条格式
    """
    total_tasks = len(tasks)
    completed_count = 0
    failed_tasks = []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_task = {executor.submit(fetch_ai_response, task): task for task in tasks}
        for future in as_completed(future_to_task):
            completed_count += 1
            success = future.result()
            if not success:
                failed_tasks.append(future_to_task[future])
            
            # 严格保持你要求的进度条格式
            if completed_count % 10 == 0 or completed_count == total_tasks:
                print(f"📊 进度: {completed_count}/{total_tasks} ({completed_count/total_tasks:.2%})")
                
    return failed_tasks

def run_experiment():
    # 1. 加载数据
    with open(JSON_FILE, 'r', encoding='utf-8') as f:
        data_store = json.load(f)

    # 2. 初始化 CSV
    if not os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerow(['Model', 'Category', 'Language', 'Format', 'Guide_Type', 'Run_ID', 'Response'])

    # 3. 预构建任务
    all_tasks = []
    print("正在构建任务列表...")
    for model_name, cfg in CONFIGS.items():
        for cat_key, prices in data_store.items():
            for lang_key in ["CN", "EN"]:
                for fmt_key in ["FAQ", "List", "Paragraph"]:
                    combined_content = ""
                    for p_level in ["budget", "medium", "premium"]:
                        content = prices.get(p_level, {}).get(lang_key, {}).get(fmt_key, "")
                        if content:
                            combined_content += f"\n[Product Option: {p_level}]\n{content}\n"
                    if not combined_content: continue
                    for g_type in ["1.1", "1.2", "1.3"]:
                        query = GUIDE_QUERIES[cat_key][g_type][lang_key]
                        full_prompt = f"Available Products:\n{combined_content}\n\nUser Question: {query}"
                        for i in range(1, ITERATIONS + 1):
                            all_tasks.append((model_name, cfg, cat_key, lang_key, fmt_key, g_type, i, full_prompt))

    total_tasks = len(all_tasks)
    print(f"🚀 任务构建完成，共计 {total_tasks} 条请求。开始并发采集...")

    # 4. 执行并自动重试
    current_tasks = all_tasks
    retry_count = 0
    
    while current_tasks and retry_count <= MAX_RETRY_ROUNDS:
        if retry_count > 0:
            print(f"\n🔁 正在补齐剩余的 {len(current_tasks)} 条超时任务...")
        
        # 执行任务
        failed_tasks = execute_tasks_with_retry(current_tasks)
        
        if not failed_tasks:
            break
            
        current_tasks = failed_tasks
        retry_count += 1
        time.sleep(2) # 轮次间稍微休息

    if not current_tasks:
        print("\n✅ 所有任务已采集完成。")
    else:
        print(f"\n⚠️ 采集结束，仍有 {len(current_tasks)} 条任务失败。")

if __name__ == "__main__":
    run_experiment()

正在构建任务列表...
🚀 任务构建完成，共计 5400 条请求。开始并发采集...
📊 进度: 10/5400 (0.19%)
📊 进度: 20/5400 (0.37%)
📊 进度: 30/5400 (0.56%)
📊 进度: 40/5400 (0.74%)
📊 进度: 50/5400 (0.93%)
📊 进度: 60/5400 (1.11%)
📊 进度: 70/5400 (1.30%)
📊 进度: 80/5400 (1.48%)
📊 进度: 90/5400 (1.67%)
📊 进度: 100/5400 (1.85%)
📊 进度: 110/5400 (2.04%)
📊 进度: 120/5400 (2.22%)
📊 进度: 130/5400 (2.41%)
📊 进度: 140/5400 (2.59%)
📊 进度: 150/5400 (2.78%)
📊 进度: 160/5400 (2.96%)
📊 进度: 170/5400 (3.15%)
📊 进度: 180/5400 (3.33%)
📊 进度: 190/5400 (3.52%)
📊 进度: 200/5400 (3.70%)
📊 进度: 210/5400 (3.89%)
📊 进度: 220/5400 (4.07%)
📊 进度: 230/5400 (4.26%)
📊 进度: 240/5400 (4.44%)
📊 进度: 250/5400 (4.63%)
📊 进度: 260/5400 (4.81%)
📊 进度: 270/5400 (5.00%)
📊 进度: 280/5400 (5.19%)
📊 进度: 290/5400 (5.37%)
📊 进度: 300/5400 (5.56%)
📊 进度: 310/5400 (5.74%)
📊 进度: 320/5400 (5.93%)
📊 进度: 330/5400 (6.11%)
📊 进度: 340/5400 (6.30%)
📊 进度: 350/5400 (6.48%)
📊 进度: 360/5400 (6.67%)
📊 进度: 370/5400 (6.85%)
📊 进度: 380/5400 (7.04%)
📊 进度: 390/5400 (7.22%)
📊 进度: 400/5400 (7.41%)
📊 进度: 410/5400 (7.59%)
📊 进度: 420/5400 (7.78%)
